# Rung 2 — Train PreyPredictor

Upload the `data/` folder (all `.npy` files) and the `quarry/` package as a Kaggle dataset.
Then run this notebook with GPU T4 accelerator.

In [ ]:
import sys, os
# Adjust these paths to match your Kaggle dataset mount
PROJECT_ROOT = '/kaggle/input/chase-project'  # where quarry/ and data/ live
sys.path.insert(0, PROJECT_ROOT)
DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
CKPT_PATH = '/kaggle/working/predictor.pt'

In [ ]:
from quarry.config import Config
from quarry.train_loop import train

cfg = Config(predictor_ckpt=CKPT_PATH)
history = train(cfg, DATA_DIR, device='cuda', epochs=50)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history['train_loss'], label='train')
axes[0].plot(history['val_loss'], label='val')
axes[0].set_title('Loss'); axes[0].legend()

axes[1].plot([m['top1_hit'] for m in history['val_metrics']])
axes[1].set_title('Val Top-1 Hit Rate')

axes[2].plot([m['argmax_cheby'] for m in history['val_metrics']])
axes[2].set_title('Val Argmax Chebyshev Dist')

plt.tight_layout()
plt.savefig('/kaggle/working/training_curves.png', dpi=150)
plt.show()

In [ ]:
# Run baselines for comparison
from quarry.dataset import split_episodes, TrajectoryDataset
from quarry.metrics import uniform_baseline, stay_baseline, last_seen_baseline
import numpy as np

_, val_eps = split_episodes(DATA_DIR, cfg.train_val_split)
ds = TrajectoryDataset(DATA_DIR, val_eps, cfg.num_hunters, cfg.history_len)
K = cfg.window_K

all_obs, all_tr, all_tc = [], [], []
for i in range(len(ds)):
    obs, tr, tc = ds[i]
    all_obs.append(obs.numpy())
    all_tr.append(tr)
    all_tc.append(tc)

tr_arr, tc_arr = np.array(all_tr), np.array(all_tc)

print('Baseline results:')
print(f"  uniform:   top1={1/(K*K):.4f}")
print(f"  stay:      {stay_baseline(tr_arr, tc_arr, K)}")
print(f"  last_seen: {last_seen_baseline(all_obs, tr_arr, tc_arr, K)}")

## Download
Download `predictor.pt` from `/kaggle/working/` and place it at `Chase/data/predictor.pt` locally.